## Categorize 211 Data

In [2]:
"""
categorize_and_chunk.py

Takes as input:
    ../data/processed/resource_pages.json

Outputs:
    ../data/processed/pages_with_categories.json
    ../data/processed/chunks.json

Steps:
1. Load pages
2. Map each page to a category/subcategory based on start_page
3. Chunk text for RAG
4. Save both JSON files
"""

'\ncategorize_and_chunk.py\n\nTakes as input:\n    ../data/processed/resource_pages.json\n\nOutputs:\n    ../data/processed/pages_with_categories.json\n    ../data/processed/chunks.json\n\nSteps:\n1. Load pages\n2. Map each page to a category/subcategory based on start_page\n3. Chunk text for RAG\n4. Save both JSON files\n'

In [4]:
import json
import os

In [4]:
# ------------------------------------------------------
# Load pages JSON
# ------------------------------------------------------
def load_pages(json_path: str):
    print(f"Loading: {json_path}")
    with open(json_path, "r", encoding="utf-8") as f:
        pages = json.load(f)
    print(f"Loaded {len(pages)} pages.")
    return pages

In [ ]:
def load_categories(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data["categories"]  # return the list

In [7]:
# ------------------------------------------------------
# Category mapping
# ------------------------------------------------------
def build_subcategory_mapping(categories):
    """Flatten category structure and compute end_page for each subcategory."""
    mapping = [] #liste vide dans laquelle on va stocker toutes les sous-catégories, chacune comme un dictionnaire.

    # Flatten all subcategories
    for cat in categories:
        for sub in cat["subcategories"]:
            mapping.append({
                "category_id": cat["category_id"],
                "category_name": cat["category_name"],
                "subcategory": sub["name"],
                "start_page": sub["start_page"]
            })

    # Sort by start page
    mapping.sort(key=lambda x: x["start_page"])

    # Add end_page
    for i in range(len(mapping)):
        if i < len(mapping) - 1:
            mapping[i]["end_page"] = mapping[i + 1]["start_page"] - 1
        else:
            mapping[i]["end_page"] = 10_000  # ending boundary
    print(f"Built mapping for {len(mapping)} subcategories.")
    print("Sample mapping entry:", mapping[0])
    print(mapping)

    return mapping


In [8]:
def assign_categories(pages, mapping):
    """Add categories to each page using start_page/end_page rules."""
    categorized = []

    for p in pages:
        page_num = p["page_number"]

        match = next(
            (m for m in mapping if m["start_page"] <= page_num <= m["end_page"]),
            None
        )

        categorized.append({
            "page_number": page_num,
            "text": p["text"],
            "category_id": match["category_id"] if match else None,
            "category_name": match["category_name"] if match else None,
            "subcategory": match["subcategory"] if match else None
        })

    print(f"Categorized {len(categorized)} pages.")
    return categorized


In [15]:
# ------------------------------------------------------
# Save JSON
# ------------------------------------------------------
def save_json(data, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"Saved → {path}")


In [ ]:
# Paths
INPUT = "../data/processed/resource_pages.json"
OUT_PAGES = "../data/processed/pages_with_categories.json"
CATEGORY_FILE = "../config/categories.json"

In [14]:
# Process
pages = load_pages(INPUT)
categories = load_categories(CATEGORY_FILE)
mapping = build_subcategory_mapping(categories)

Loading: ../data/processed/resource_pages.json
Loaded 3090 pages.
Built mapping for 159 subcategories.
Sample mapping entry: {'category_id': 17, 'category_name': 'Sports, Recreation and Culture', 'subcategory': 'Sports and recreation for people with a disability', 'start_page': 0, 'end_page': 1}
[{'category_id': 17, 'category_name': 'Sports, Recreation and Culture', 'subcategory': 'Sports and recreation for people with a disability', 'start_page': 0, 'end_page': 1}, {'category_id': 1, 'category_name': 'Child, Youth and Family', 'subcategory': 'Child welfare', 'start_page': 2, 'end_page': 11}, {'category_id': 1, 'category_name': 'Child, Youth and Family', 'subcategory': 'Daycare and respite', 'start_page': 12, 'end_page': 35}, {'category_id': 1, 'category_name': 'Child, Youth and Family', 'subcategory': 'Family Support', 'start_page': 36, 'end_page': 95}, {'category_id': 1, 'category_name': 'Child, Youth and Family', 'subcategory': 'Information on childcare services', 'start_page': 96, 

In [15]:
categorized_pages = assign_categories(pages, mapping)


Categorized 3090 pages.


In [17]:
save_json(categorized_pages, OUT_PAGES)

Saved → ../data/processed/pages_with_categories.json


## Categorize macmmunaute Data

In [8]:
# Paths
INPUT = "../data/processed/macommunaute/"
OUT_PAGES = "../data/processed/macommunaute/"
CATEGORY_FILE = "../config/categories.json"

In [10]:
# Map file names to category names from categories.json
FILE_TO_CATEGORY = {
    "alimentation": "Food",
    "aide_ecoute": "Mental Health and addictions",
    "communautes_culturelles_nouveaux_arrivants": "Immigration and cultural communities",
    "education": "Education and Literacy",
    "emploi": "Employment and Income Support",
    "famille": "Child, Youth and Family",
    "justice_droits": "Justice and Advocacy",
    "logement": "Housing",
    "retraite": "Seniors",
    "sante": "Health",
    "sports_loisirs": "Sports, Recreation and Culture"
}

def find_category_by_name(category_name, categories):
    """Find category ID from categories list by name."""
    for cat in categories:
        if cat["category_name"] == category_name:
            return cat["category_id"], cat["category_name"]
    return None, None

def get_category_for_file(filename, categories):
    """Determine category for a file based on its name."""
    # Remove extension and language suffix (e.g., "_en.json")
    base_name = filename.replace(".json", "").replace("_en", "")
    
    # Find matching category
    for file_pattern, category_name in FILE_TO_CATEGORY.items():
        if base_name == file_pattern:
            cat_id, cat_name = find_category_by_name(category_name, categories)
            return cat_id, cat_name
    
    return None, None

In [12]:
def categorize_macommunaute_pages(pages, category_id, category_name):
    """Assign all pages in a file to the same category."""
    categorized = []
    
    for p in pages:
        categorized.append({
            "page_number": p["page_number"],
            "text": p["text"],
            "category_id": category_id,
            "category_name": category_name,
            "subcategory": ""
        })
    
    return categorized

In [13]:
# Get all JSON files in the macommunaute directory
import glob
# Find all JSON files
json_files = glob.glob(os.path.join(INPUT, "*.json"))
print(f"Found {len(json_files)} JSON files in {INPUT}")
for f in json_files:
    print(f"  - {os.path.basename(f)}")

Found 11 JSON files in ../data/processed/macommunaute/
  - sports_loisirs_en.json
  - sante_en.json
  - retraite_en.json
  - logement_en.json
  - justice_droits_en.json
  - emploi_en.json
  - education_en.json
  - alimentation_en.json
  - aide_ecoute_en.json
  - communautes_culturelles_nouveaux_arrivants_en.json
  - famille_en.json


In [16]:
# Process each macommunaute file
import glob

categories = load_categories(CATEGORY_FILE)
output_dir = "../data/processed/macommunaute/"

json_files = glob.glob(os.path.join(INPUT, "*.json"))
print(f"Found {len(json_files)} JSON files\n")

for json_file in json_files:
    filename = os.path.basename(json_file)
    print(f"{'='*60}")
    print(f"Processing: {filename}")
    print(f"{'='*60}")
    
    # Determine category from filename
    category_id, category_name = get_category_for_file(filename, categories)
    
    if category_id is None:
        print(f"⚠ WARNING: Could not find category for {filename}")
        print(f"  Base name pattern not found in FILE_TO_CATEGORY mapping")
        continue
    
    print(f"  Category: {category_name} (ID: {category_id})")
    
    # Load pages
    pages = load_pages(json_file)
    
    # Categorize all pages to this category
    categorized_pages = categorize_macommunaute_pages(pages, category_id, category_name)
    
    # Save to output directory
    output_filename = filename.replace(".json", "_categorized.json")
    output_path = os.path.join(output_dir, output_filename)
    save_json(categorized_pages, output_path)
    
    print(f"✓ Saved {len(categorized_pages)} categorized pages to {output_path}\n")

print(f"{'='*60}")
print(f"✓ All files processed successfully!")
print(f"{'='*60}")

Found 11 JSON files

Processing: sports_loisirs_en.json
  Category: Sports, Recreation and Culture (ID: 17)
Loading: ../data/processed/macommunaute\sports_loisirs_en.json
Loaded 37 pages.
Saved → ../data/processed/macommunaute/sports_loisirs_en_categorized.json
✓ Saved 37 categorized pages to ../data/processed/macommunaute/sports_loisirs_en_categorized.json

Processing: sante_en.json
  Category: Health (ID: 8)
Loading: ../data/processed/macommunaute\sante_en.json
Loaded 65 pages.
Saved → ../data/processed/macommunaute/sante_en_categorized.json
✓ Saved 65 categorized pages to ../data/processed/macommunaute/sante_en_categorized.json

Processing: retraite_en.json
  Category: Seniors (ID: 15)
Loading: ../data/processed/macommunaute\retraite_en.json
Loaded 1 pages.
Saved → ../data/processed/macommunaute/retraite_en_categorized.json
✓ Saved 1 categorized pages to ../data/processed/macommunaute/retraite_en_categorized.json

Processing: logement_en.json
  Category: Housing (ID: 10)
Loading: ..

In [18]:
# Verification: Show summary of categorized files
print(f"\n{'='*60}")
print(f"Verification Summary")
print(f"{'='*60}\n")

output_files = glob.glob(os.path.join(output_dir, "*_categorized.json"))
print(f"Created {len(output_files)} categorized files:\n")

for output_file in sorted(output_files):
    with open(output_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    if data:
        sample = data[0]
        print(f"✓ {os.path.basename(output_file)}")
        print(f"  Pages: {len(data)}")
        print(f"  Category: {sample['category_name']} (ID: {sample['category_id']})")
        print(f"  Subcategory: '{sample['subcategory']}'")
        print()


Verification Summary

Created 11 categorized files:

✓ aide_ecoute_en_categorized.json
  Pages: 32
  Category: Mental Health and addictions (ID: 14)
  Subcategory: ''

✓ alimentation_en_categorized.json
  Pages: 7
  Category: Food (ID: 6)
  Subcategory: ''

✓ communautes_culturelles_nouveaux_arrivants_en_categorized.json
  Pages: 40
  Category: Immigration and cultural communities (ID: 11)
  Subcategory: ''

✓ education_en_categorized.json
  Pages: 34
  Category: Education and Literacy (ID: 4)
  Subcategory: ''

✓ emploi_en_categorized.json
  Pages: 8
  Category: Employment and Income Support (ID: 5)
  Subcategory: ''

✓ famille_en_categorized.json
  Pages: 54
  Category: Child, Youth and Family (ID: 1)
  Subcategory: ''

✓ justice_droits_en_categorized.json
  Pages: 7
  Category: Justice and Advocacy (ID: 13)
  Subcategory: ''

✓ logement_en_categorized.json
  Pages: 17
  Category: Housing (ID: 10)
  Subcategory: ''

✓ retraite_en_categorized.json
  Pages: 1
  Category: Seniors (ID: 